In [39]:
!pip install rlcard

1164.82s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [99]:
import rlcard
from rlcard.agents import DQNAgent
from rlcard import models
from rlcard.utils.utils import set_seed
from rlcard.utils import tournament
from rlcard.utils import Logger
from rlcard.agents import RandomAgent
from rlcard.utils.utils import reorganize

In [41]:
# Set a global seed
set_seed(0)

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


In [42]:
# Create the environment
env = rlcard.make('no-limit-holdem')


In [43]:
# Set the hyperparameters
agent_config = {
    'seed': 0,
    'train_every': 1,
    'evaluate_every': 10,
    'memory_init_size': 1000,
    'train_target_every': 100,
    'eps_decay_rate': 0.99,
    'eps_min': 0.1,
    'num_eval_games': 1000,
    'rl_learning_rate': 0.0001,
    'rl_target_update_rate': 0.1,
    'rl_memory_size': 10000,
    'rl_batch_size': 32,
    'rl_train_epoch': 1,
    'hidden_layers_sizes': [512, 512]
}


In [44]:
agent = DQNAgent(num_actions=env.num_actions,
                 replay_memory_init_size=1000,
                 train_every=1,
                 state_shape=env.state_shape,
                 mlp_layers=[128, 128])

In [69]:
# # Train the agent
# for episode in range(1000): #num_episodes
#     # Reset the environment and get the initial state
#     state = env.reset()
    
#     while True:
#         # Choose an action based on the current state
#         action = agent.step(state)
        
#         # Take the chosen action and get the next state, reward, and done flag
#         next_state, reward, done = env.step(action)
        
#         # Store the transition in the agent's replay memory
#         agent.feed(state, action, reward, next_state, done)
        
#         # Update the agent's Q-network and target network using experience replay
#         agent.train()
        
#         # Update the current state
#         state = next_state
        
#         if done:
#             break

In [50]:
import rlcard
from rlcard.agents import DQNAgent
from rlcard.utils import tournament
from rlcard.utils import Logger
import numpy as np
import torch
import random

def set_seed(seed=0):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

env = rlcard.make('limit-holdem', config={'seed': 0})
eval_env = rlcard.make('limit-holdem', config={'seed': 0})

set_seed(0)

agent = DQNAgent(num_actions=env.num_actions, 
                 replay_memory_size=50000, 
                 update_target_estimator_every=1000, 
                 state_shape= [72],
                 discount_factor=0.99, 
                 epsilon_start=1.0, 
                 epsilon_end=0.1, 
                 epsilon_decay_steps=20000, 
                 batch_size=32, 
                 train_every=1, 
                 mlp_layers=[64, 64])
env.set_agents([agent, agent])

log_dir = './experiments/limit_holdem_dqn_result/'
logger = Logger(log_dir)

for episode in range(10000):
    trajectories, payoffs = env.run(is_training=True)
    
    agent.feed(trajectories)
    if episode % 100 == 0:
        logger.log_performance(env.tournament(eval_env, 1000)[0])

logger.close_files()

[[{'legal_actions': OrderedDict([(0, None), (1, None), (2, None)]), 'obs': array([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1.,
       0., 0., 0., 0.]), 'raw_obs': {'hand': ['CT', 'S7'], 'public_cards': [], 'all_chips': [1, 2], 'my_chips': 1, 'legal_actions': ['call', 'raise', 'fold'], 'raise_nums': [0, 0, 0, 0]}, 'raw_legal_actions': ['call', 'raise', 'fold'], 'action_record': [(0, 'raise'), (1, 'fold')]}, 1, {'legal_actions': OrderedDict([(1, None), (2, None), (3, None)]), 'obs': array([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 1., 0., 0., 0., 1., 0., 0.

ValueError: not enough values to unpack (expected 5, got 2)

In [105]:
import rlcard
from rlcard.agents import DQNAgent
from rlcard.utils import Logger
from rlcard.utils.utils import tournament
import numpy as np
import torch
import random
import json

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# Create the environment
env = rlcard.make('no-limit-holdem', config={'seed': 0, 'num_players':4, 'initial_chips':1000, 'allow_step_back':False, 'seed':69})
eval_env = rlcard.make('no-limit-holdem', config={'seed': 0, 'num_players':4, 'initial_chips':1000, 'allow_step_back':False, 'seed':69})

# Set random seed
set_seed(42)

# Create the DQN asgent
agent = DQNAgent(num_actions=env.num_actions, 
                 replay_memory_size=50000, 
                 update_target_estimator_every=1000, 
                 state_shape=[54],
                 discount_factor=0.99, 
                 epsilon_start=1.0, 
                 epsilon_end=0.1, 
                 epsilon_decay_steps=20000, 
                 batch_size=32, 
                 train_every=1, 
                 mlp_layers=[64, 64])

newagent = RandomAgent(num_actions=env.num_actions)

# Set the agents for the environment
env.set_agents([agent, newagent, agent, newagent, agent])

# Create a logger
log_dir = './experiments/no_limit_holdem_dqn_result/'
logger = Logger(log_dir)

# Training loop
for episode in range(20000):
    # Generate trajectories
    trajectories, payoffs = env.run(is_training=True)
    newtraj = reorganize(trajectories, payoffs)
    # print(payoffs)
    # input("")
    # Feed trajectories to the DQN agent
    for ts in newtraj[0]:
            agent.feed(ts)

    # Log performance
    if episode % 100 == 0:
        logger.log_performance(tournament(eval_env, 1000)[0])


# Close the logger
logger.close_files()

AttributeError: 'NolimitholdemEnv' object has no attribute 'agents'

In [37]:
# # Set the game parameters
# num_episodes = 10000
# num_steps = 100
# learning_rate = 0.0005


In [36]:
# # Initialize the environment
# env = rlcard.make('no-limit-holdem')

# # Initialize the DQN agent
# agent = DQNAgent(
#     num_actions=env.num_actions,
#     state_shape=[1],  # Specify state shape
#     replay_memory_size=50000,
#     update_target_estimator_every=1000,
#     discount_factor=0.99,
#     epsilon_start=1.0,
#     epsilon_end=0.1,
#     epsilon_decay_steps=20000,
#     batch_size=32,
#     train_every=1,
#     mlp_layers=[64, 64]
# )

# # Initialize the random agent for benchmarking
# random_agent = RandomAgent(num_actions=env.num_actions)


In [35]:
# # Initialize the logger
# logger = Logger('./experiments/dqn_self_play')
# logger.log_params({
#     'num_episodes': num_episodes,
#     'num_steps': num_steps,
#     'learning_rate': learning_rate
# })

In [66]:
# import unittest
# import torch
# import numpy as np

# from rlcard.agents.dqn_agent import DQNAgent

# class TestDQN(unittest.TestCase):

#     def test_init(self):

#         agent = DQNAgent(replay_memory_size=0,
#                          replay_memory_init_size=0,
#                          update_target_estimator_every=0,
#                          discount_factor=0,
#                          epsilon_start=0,
#                          epsilon_end=0,
#                          epsilon_decay_steps=0,
#                          batch_size=0,
#                          num_actions=2,
#                          state_shape=[1],
#                          mlp_layers=[10,10],
#                          device=torch.device('cpu'))

#         self.assertEqual(agent.replay_memory_init_size, 0)
#         self.assertEqual(agent.update_target_estimator_every, 0)
#         self.assertEqual(agent.discount_factor, 0)
#         self.assertEqual(agent.epsilon_decay_steps, 0)
#         self.assertEqual(agent.batch_size, 0)
#         self.assertEqual(agent.num_actions, 2)

#     def test_train(self):

#         memory_init_size = 100
#         num_steps = 500

#         agent = DQNAgent(replay_memory_size = 200,
#                          replay_memory_init_size=memory_init_size,
#                          update_target_estimator_every=100,
#                          state_shape=[2],
#                          mlp_layers=[10,10],
#                          device=torch.device('cpu'))

#         predicted_action, _ = agent.eval_step({'obs': np.random.random_sample((2,)), 'legal_actions': {0: None, 1: None}, 'raw_legal_actions': ['call', 'raise']})
#         self.assertGreaterEqual(predicted_action, 0)
#         self.assertLessEqual(predicted_action, 1)

#         for _ in range(num_steps):
#             ts = [{'obs': np.random.random_sample((2,)), 'legal_actions': {0: None, 1: None}}, np.random.randint(2), 0, {'obs': np.random.random_sample((2,)), 'legal_actions': {0: None, 1: None}, 'raw_legal_actions': ['call', 'raise']}, True]
#             agent.feed(ts)

#         predicted_action = agent.step({'obs': np.random.random_sample((2,)), 'legal_actions': {0: None, 1: None}})
#         self.assertGreaterEqual(predicted_action, 0)
#         self.assertLessEqual(predicted_action, 1)